# Project 01 — Retail Data Cleaning and Customer Analysis

**Author:** Jorgo Luka  
**Format:** Standalone Google Colab notebook  
**Dataset:** UCI Online Retail — 541,909 transactions from a UK online retailer  
**Core tools:** Python · Pandas · NumPy · Matplotlib · Seaborn

## Recruiter summary

This project turns a messy transaction ledger into audited analytical tables and decision-ready customer insights. It does not hide inconvenient rows or treat deletion as “cleaning.” Every exclusion is assigned a reason, reconciled to the raw data and exported for review.

| Capability | Evidence |
|---|---|
| Data cleaning | Explicit contract, duplicate handling, cancellations, returns, missing values and invalid-price rules |
| Data analysis | Monthly KPIs, product/country performance, customer RFM and cohort retention |
| Engineering | Deterministic pipeline, assertions, fixture tests, audit report, artifacts and SHA-256 manifest |
| Communication | Executive summary, documented limitations and customer lookup interface |

> The dataset is historical and anonymised. Results demonstrate analytical technique; they are not claims about a current retailer.

## Business questions

1. What is the exact condition of the raw transaction ledger?
2. Which rows are completed sales, cancellations, returns, adjustments or unusable records?
3. How do revenue, orders and customers change by month?
4. Which products and countries contribute the most revenue?
5. Which customers are recent, frequent and valuable?
6. How well do acquisition cohorts retain over subsequent months?

## Data source and licence

- Daqing Chen (2015), **Online Retail**, UCI Machine Learning Repository.
- DOI: [10.24432/C5BW33](https://doi.org/10.24432/C5BW33)
- Dataset page: [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail)
- Licence: Creative Commons Attribution 4.0.

The source contains transactions from 1 December 2010 to 9 December 2011. Invoice numbers beginning with `C` indicate cancellations. Prices are in pounds sterling.

## 0. Environment and reproducibility

Run this notebook from a fresh Colab runtime. No API key or private account is required.

In [ ]:
%pip -q install "openpyxl>=3.1" "pyarrow>=15" "gradio>=5,<7"

In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import platform
import random
import re
import shutil
import urllib.request
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable, Mapping

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


@dataclass(frozen=True)
class ProjectConfig:
    dataset_url: str = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
    expected_workbook: str = "Online Retail.xlsx"
    artifact_dir: str = "retail_analysis_artifacts"
    anomaly_threshold: float = 4.0
    top_n: int = 12
    launch_app: bool = False


CFG = ProjectConfig()
ARTIFACTS = Path(CFG.artifact_dir)
ARTIFACTS.mkdir(exist_ok=True)

print({
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "configuration": asdict(CFG),
})

## 1. Acquire and fingerprint the source data

The workbook is downloaded from UCI, extracted locally and fingerprinted. Recording the hash makes silent source changes visible in later runs.

In [ ]:
def sha256_file(path: str | Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def acquire_workbook(config: ProjectConfig = CFG) -> tuple[Path, dict[str, Any]]:
    data_dir = Path("retail_source_data")
    data_dir.mkdir(exist_ok=True)
    archive_path = data_dir / "online_retail.zip"
    workbook_path = data_dir / config.expected_workbook
    if not workbook_path.exists():
        print("Downloading official UCI archive …")
        urllib.request.urlretrieve(config.dataset_url, archive_path)
        with zipfile.ZipFile(archive_path) as archive:
            members = archive.namelist()
            if config.expected_workbook not in members:
                raise FileNotFoundError(f"Expected {config.expected_workbook}; found {members}")
            archive.extract(config.expected_workbook, data_dir)
    if workbook_path.stat().st_size < 10_000_000:
        raise ValueError("Workbook is unexpectedly small; delete it and rerun the acquisition cell")
    fingerprint = {
        "source_url": config.dataset_url,
        "workbook": str(workbook_path),
        "bytes": workbook_path.stat().st_size,
        "sha256": sha256_file(workbook_path),
        "retrieved_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    }
    return workbook_path, fingerprint


WORKBOOK, SOURCE_FINGERPRINT = acquire_workbook()
print(json.dumps(SOURCE_FINGERPRINT, indent=2))

## 2. Load the raw ledger and enforce a data contract

The contract verifies schema, parseability and scale before any business calculation is allowed.

In [ ]:
EXPECTED_COLUMNS = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country",
]


def load_raw_retail(path: str | Path) -> pd.DataFrame:
    frame = pd.read_excel(path, engine="openpyxl")
    missing = sorted(set(EXPECTED_COLUMNS) - set(frame.columns))
    unexpected = sorted(set(frame.columns) - set(EXPECTED_COLUMNS))
    if missing or unexpected:
        raise ValueError({"missing_columns": missing, "unexpected_columns": unexpected})
    frame = frame[EXPECTED_COLUMNS].copy()
    frame["InvoiceDate"] = pd.to_datetime(frame["InvoiceDate"], errors="coerce")
    frame["Quantity"] = pd.to_numeric(frame["Quantity"], errors="coerce")
    frame["UnitPrice"] = pd.to_numeric(frame["UnitPrice"], errors="coerce")
    if len(frame) < 500_000:
        raise ValueError(f"Expected at least 500,000 rows, received {len(frame):,}")
    return frame


raw = load_raw_retail(WORKBOOK)
print(f"Raw shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
display(raw.head())
display(raw.dtypes.rename("dtype").to_frame())

## 3. Profile quality before changing anything

Missing customer identifiers are not automatically errors: anonymous sales remain useful for transaction-level analysis but cannot enter customer-level analysis. Cancellations and returns are retained in the financial ledger and excluded only from completed-sales views.

In [ ]:
def quality_profile(frame: pd.DataFrame) -> pd.DataFrame:
    invoice = frame["InvoiceNo"].astype("string").str.strip().str.upper()
    checks = {
        "rows": len(frame),
        "columns": frame.shape[1],
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "missing_descriptions": int(frame["Description"].isna().sum()),
        "missing_customer_ids": int(frame["CustomerID"].isna().sum()),
        "invalid_dates": int(frame["InvoiceDate"].isna().sum()),
        "cancellation_rows": int(invoice.str.startswith("C", na=False).sum()),
        "negative_quantity_rows": int(frame["Quantity"].lt(0).sum()),
        "zero_quantity_rows": int(frame["Quantity"].eq(0).sum()),
        "negative_price_rows": int(frame["UnitPrice"].lt(0).sum()),
        "zero_price_rows": int(frame["UnitPrice"].eq(0).sum()),
        "unique_invoices": int(invoice.nunique()),
        "unique_products": int(frame["StockCode"].nunique()),
        "unique_countries": int(frame["Country"].nunique()),
    }
    result = pd.Series(checks, name="value").to_frame()
    result["percent_of_rows"] = result["value"] / max(len(frame), 1) * 100
    return result


raw_quality = quality_profile(raw)
display(raw_quality.style.format({"value": "{:,.0f}", "percent_of_rows": "{:.2f}%"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
raw.isna().mean().sort_values().mul(100).plot.barh(ax=axes[0], color="#4c78a8")
axes[0].set(title="Missing values by column", xlabel="missing (%)", ylabel="")
status_counts = pd.Series({
    "completed quantity": raw["Quantity"].gt(0).sum(),
    "negative quantity": raw["Quantity"].lt(0).sum(),
    "non-positive price": raw["UnitPrice"].le(0).sum(),
    "duplicate": raw.duplicated().sum(),
}).sort_values()
status_counts.plot.barh(ax=axes[1], color="#f58518")
axes[1].set(title="Primary quality signals", xlabel="rows", ylabel="")
plt.tight_layout()
plt.show()

## 4. Build an auditable cleaning pipeline

The pipeline produces three separate tables:

- `ledger`: deduplicated source rows with status flags and signed line value;
- `completed_sales`: valid positive sales for product and revenue analysis;
- `customer_sales`: completed sales with a known customer identifier.

This separation prevents customer analysis rules from silently changing company-level revenue.

In [ ]:
def normalise_identifier(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.upper()


def clean_retail_data(frame: pd.DataFrame) -> dict[str, Any]:
    working = frame.copy()
    raw_rows = len(working)
    duplicate_mask = working.duplicated(keep="first")
    duplicates = working.loc[duplicate_mask].copy()
    working = working.loc[~duplicate_mask].copy()

    working.columns = [
        "invoice_no", "stock_code", "description", "quantity",
        "invoice_date", "unit_price", "customer_id", "country",
    ]
    working["invoice_no"] = normalise_identifier(working["invoice_no"])
    working["stock_code"] = normalise_identifier(working["stock_code"])
    working["description"] = working["description"].astype("string").str.strip().str.upper()
    working["country"] = working["country"].astype("string").str.strip()
    working["invoice_date"] = pd.to_datetime(working["invoice_date"], errors="coerce")
    numeric_customer = pd.to_numeric(working["customer_id"], errors="coerce")
    working["customer_id"] = numeric_customer.round().astype("Int64").astype("string")

    working["is_cancellation"] = working["invoice_no"].str.startswith("C", na=False)
    working["is_return"] = working["quantity"].lt(0)
    working["has_valid_date"] = working["invoice_date"].notna()
    working["has_description"] = working["description"].notna() & working["description"].ne("")
    working["has_positive_price"] = working["unit_price"].gt(0)
    working["has_positive_quantity"] = working["quantity"].gt(0)
    working["has_customer_id"] = working["customer_id"].notna()
    working["line_value_gbp"] = working["quantity"] * working["unit_price"]

    conditions = [
        ~working["has_valid_date"],
        ~working["has_description"],
        ~working["has_positive_price"],
        working["is_cancellation"],
        working["is_return"],
        ~working["has_positive_quantity"],
    ]
    labels = [
        "invalid_date", "missing_description", "non_positive_price",
        "cancellation", "return_or_negative_quantity", "non_positive_quantity",
    ]
    working["row_status"] = np.select(conditions, labels, default="completed_sale")

    financial_ledger = working.loc[
        working["has_valid_date"]
        & working["has_description"]
        & working["has_positive_price"]
        & working["quantity"].ne(0)
    ].copy()
    completed_sales = working.loc[working["row_status"].eq("completed_sale")].copy()
    customer_sales = completed_sales.loc[completed_sales["has_customer_id"]].copy()

    reconciliation = pd.DataFrame({
        "measure": [
            "raw rows", "duplicates removed", "deduplicated ledger rows",
            "financial ledger rows", "completed sales rows", "customer sales rows",
        ],
        "rows": [
            raw_rows, len(duplicates), len(working), len(financial_ledger),
            len(completed_sales), len(customer_sales),
        ],
    })
    return {
        "ledger": working,
        "financial_ledger": financial_ledger,
        "completed_sales": completed_sales,
        "customer_sales": customer_sales,
        "duplicates": duplicates,
        "reconciliation": reconciliation,
    }


cleaned = clean_retail_data(raw)
ledger = cleaned["ledger"]
financial_ledger = cleaned["financial_ledger"]
completed_sales = cleaned["completed_sales"]
customer_sales = cleaned["customer_sales"]
reconciliation = cleaned["reconciliation"]

display(reconciliation.style.format({"rows": "{:,.0f}"}))
display(ledger["row_status"].value_counts().rename_axis("status").to_frame("rows"))

## 5. Prove the cleaning rules with a controlled fixture

This small fixture tests the decisions that matter: exact duplicates, cancellations, zero-price rows and anonymous customers.

In [ ]:
fixture = pd.DataFrame([
    ["100001", "12345", "VALID PRODUCT", 2, "2011-01-01", 10.0, 12345, "UK"],
    ["100001", "12345", "VALID PRODUCT", 2, "2011-01-01", 10.0, 12345, "UK"],
    ["C100001", "12345", "VALID PRODUCT", -1, "2011-01-02", 10.0, 12345, "UK"],
    ["100002", "54321", "FREE ITEM", 1, "2011-01-03", 0.0, 12346, "UK"],
    ["100003", "99999", "ANONYMOUS SALE", 1, "2011-01-04", 5.0, np.nan, "UK"],
], columns=EXPECTED_COLUMNS)
fixture["InvoiceDate"] = pd.to_datetime(fixture["InvoiceDate"])
fixture_result = clean_retail_data(fixture)

assert len(fixture_result["duplicates"]) == 1
assert len(fixture_result["ledger"]) == 4
assert fixture_result["ledger"]["row_status"].value_counts().to_dict() == {
    "completed_sale": 2,
    "cancellation": 1,
    "non_positive_price": 1,
}
assert len(fixture_result["completed_sales"]) == 2
assert len(fixture_result["customer_sales"]) == 1
print("Controlled cleaning fixture passed.")

## 6. Reconcile financial totals and monthly performance

Gross sales use completed positive transactions. Net ledger value also includes valid cancellations and returns, retaining their negative sign.

In [ ]:
gross_revenue = float(completed_sales["line_value_gbp"].sum())
net_ledger_revenue = float(financial_ledger["line_value_gbp"].sum())
refund_value = float(-financial_ledger.loc[financial_ledger["line_value_gbp"].lt(0), "line_value_gbp"].sum())
cancellation_invoices = int(ledger.loc[ledger["is_cancellation"], "invoice_no"].nunique())
all_invoices = int(ledger["invoice_no"].nunique())

headline_kpis = {
    "raw_rows": len(raw),
    "completed_sales_rows": len(completed_sales),
    "unique_completed_orders": int(completed_sales["invoice_no"].nunique()),
    "known_customers": int(customer_sales["customer_id"].nunique()),
    "gross_revenue_gbp": gross_revenue,
    "net_ledger_revenue_gbp": net_ledger_revenue,
    "refund_and_return_value_gbp": refund_value,
    "cancellation_invoice_rate": cancellation_invoices / all_invoices,
}
display(pd.Series(headline_kpis, name="value").to_frame())

sales_monthly = (
    completed_sales.assign(month=completed_sales["invoice_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month")
    .agg(
        gross_revenue_gbp=("line_value_gbp", "sum"),
        orders=("invoice_no", "nunique"),
        active_customers=("customer_id", "nunique"),
        units=("quantity", "sum"),
    )
    .reset_index()
)
sales_monthly["average_order_value_gbp"] = (
    sales_monthly["gross_revenue_gbp"] / sales_monthly["orders"]
)
display(sales_monthly)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.lineplot(data=sales_monthly, x="month", y="gross_revenue_gbp", marker="o", ax=axes[0, 0], color="#2563eb")
axes[0, 0].set(title="Monthly gross revenue", xlabel="", ylabel="GBP")
sns.lineplot(data=sales_monthly, x="month", y="orders", marker="o", ax=axes[0, 1], color="#059669")
axes[0, 1].set(title="Monthly completed orders", xlabel="", ylabel="orders")
sns.lineplot(data=sales_monthly, x="month", y="active_customers", marker="o", ax=axes[1, 0], color="#7c3aed")
axes[1, 0].set(title="Monthly active known customers", xlabel="", ylabel="customers")
sns.lineplot(data=sales_monthly, x="month", y="average_order_value_gbp", marker="o", ax=axes[1, 1], color="#dc2626")
axes[1, 1].set(title="Average order value", xlabel="", ylabel="GBP")
for axis in axes.flat:
    axis.tick_params(axis="x", rotation=35)
plt.tight_layout()
monthly_figure = ARTIFACTS / "monthly_performance.png"
plt.savefig(monthly_figure, dpi=170, bbox_inches="tight")
plt.show()

## 7. Product and country performance

Top lists use completed sales only. The United Kingdom is shown separately from export markets because its scale otherwise hides international variation.

In [ ]:
product_performance = (
    completed_sales.groupby(["stock_code", "description"], dropna=False)
    .agg(revenue_gbp=("line_value_gbp", "sum"), units=("quantity", "sum"), orders=("invoice_no", "nunique"))
    .reset_index()
    .sort_values("revenue_gbp", ascending=False)
)
country_performance = (
    completed_sales.groupby("country", dropna=False)
    .agg(revenue_gbp=("line_value_gbp", "sum"), orders=("invoice_no", "nunique"), customers=("customer_id", "nunique"))
    .reset_index()
    .sort_values("revenue_gbp", ascending=False)
)
international = country_performance.loc[country_performance["country"].ne("United Kingdom")].head(CFG.top_n)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.barplot(data=product_performance.head(CFG.top_n), x="revenue_gbp", y="description", ax=axes[0], color="#2563eb")
axes[0].set(title="Top products by completed-sales revenue", xlabel="GBP", ylabel="")
sns.barplot(data=international, x="revenue_gbp", y="country", ax=axes[1], color="#059669")
axes[1].set(title="Top non-UK markets by revenue", xlabel="GBP", ylabel="")
plt.tight_layout()
product_country_figure = ARTIFACTS / "product_and_country_performance.png"
plt.savefig(product_country_figure, dpi=170, bbox_inches="tight")
plt.show()

display(product_performance.head(CFG.top_n))
display(country_performance.head(CFG.top_n))

## 8. Customer RFM analysis

RFM summarises **recency**, **frequency** and **monetary value**. Scores are descriptive prioritisation aids, not predictions of customer behaviour.

In [ ]:
def quintile_score(series: pd.Series, higher_is_better: bool = True) -> pd.Series:
    percentile = series.rank(method="average", pct=True)
    score = np.ceil(percentile * 5).clip(1, 5).astype(int)
    return score if higher_is_better else 6 - score


reference_date = customer_sales["invoice_date"].max().normalize() + pd.Timedelta(days=1)
rfm = (
    customer_sales.groupby("customer_id")
    .agg(
        last_purchase=("invoice_date", "max"),
        frequency=("invoice_no", "nunique"),
        monetary_gbp=("line_value_gbp", "sum"),
        units=("quantity", "sum"),
        country=("country", lambda values: values.mode().iat[0] if not values.mode().empty else pd.NA),
    )
    .reset_index()
)
rfm["recency_days"] = (reference_date - rfm["last_purchase"].dt.normalize()).dt.days
rfm["r_score"] = quintile_score(rfm["recency_days"], higher_is_better=False)
rfm["f_score"] = quintile_score(rfm["frequency"], higher_is_better=True)
rfm["m_score"] = quintile_score(rfm["monetary_gbp"], higher_is_better=True)
rfm["rfm_total"] = rfm[["r_score", "f_score", "m_score"]].sum(axis=1)

conditions = [
    rfm[["r_score", "f_score", "m_score"]].ge(4).all(axis=1),
    rfm["r_score"].ge(3) & rfm["f_score"].ge(4),
    rfm["r_score"].ge(4) & rfm["f_score"].le(3),
    rfm["r_score"].le(2) & rfm["f_score"].ge(3),
    rfm["r_score"].le(2) & rfm["f_score"].le(2),
]
segments = ["Champions", "Loyal", "Potential", "At risk", "Hibernating"]
rfm["segment"] = np.select(conditions, segments, default="Needs attention")

segment_summary = (
    rfm.groupby("segment")
    .agg(
        customers=("customer_id", "nunique"),
        median_recency_days=("recency_days", "median"),
        median_frequency=("frequency", "median"),
        total_revenue_gbp=("monetary_gbp", "sum"),
    )
    .reset_index()
    .sort_values("total_revenue_gbp", ascending=False)
)
segment_summary["customer_share"] = segment_summary["customers"] / segment_summary["customers"].sum()
segment_summary["revenue_share"] = segment_summary["total_revenue_gbp"] / segment_summary["total_revenue_gbp"].sum()
display(segment_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.barplot(data=segment_summary, x="customers", y="segment", ax=axes[0], color="#2563eb")
axes[0].set(title="Customers by RFM segment", xlabel="customers", ylabel="")
sns.barplot(data=segment_summary, x="total_revenue_gbp", y="segment", ax=axes[1], color="#f59e0b")
axes[1].set(title="Historical revenue by RFM segment", xlabel="GBP", ylabel="")
plt.tight_layout()
segment_figure = ARTIFACTS / "customer_segments.png"
plt.savefig(segment_figure, dpi=170, bbox_inches="tight")
plt.show()

## 9. Acquisition cohort retention

Retention counts a customer as active when they place at least one completed order in a later calendar month. The final cohort is incomplete because the dataset ends on 9 December 2011.

In [ ]:
orders = customer_sales[["customer_id", "invoice_no", "invoice_date"]].drop_duplicates().copy()
orders["order_month"] = orders["invoice_date"].dt.to_period("M").dt.to_timestamp()
orders["cohort_month"] = orders.groupby("customer_id")["order_month"].transform("min")
orders["cohort_index"] = (
    (orders["order_month"].dt.year - orders["cohort_month"].dt.year) * 12
    + orders["order_month"].dt.month
    - orders["cohort_month"].dt.month
    + 1
)

cohort_counts = (
    orders.groupby(["cohort_month", "cohort_index"])["customer_id"]
    .nunique()
    .unstack(fill_value=0)
    .sort_index()
)
cohort_sizes = cohort_counts[1]
cohort_retention = cohort_counts.divide(cohort_sizes, axis=0)

fig, ax = plt.subplots(figsize=(15, 8))
sns.heatmap(cohort_retention, cmap="Blues", vmin=0, vmax=1, annot=True, fmt=".0%", ax=ax)
ax.set(title="Monthly customer retention by acquisition cohort", xlabel="months since first purchase", ylabel="cohort month")
ax.set_yticklabels([value.strftime("%Y-%m") for value in cohort_retention.index], rotation=0)
plt.tight_layout()
cohort_figure = ARTIFACTS / "cohort_retention.png"
plt.savefig(cohort_figure, dpi=170, bbox_inches="tight")
plt.show()

## 10. Robust anomaly monitoring

Daily net ledger value is monitored using a median absolute deviation score. Flags are prompts for investigation—not proof of fraud or operational failure.

In [ ]:
daily = (
    financial_ledger.assign(date=financial_ledger["invoice_date"].dt.normalize())
    .groupby("date")
    .agg(net_revenue_gbp=("line_value_gbp", "sum"), rows=("invoice_no", "size"), invoices=("invoice_no", "nunique"))
    .reset_index()
)
median_revenue = daily["net_revenue_gbp"].median()
mad = np.median(np.abs(daily["net_revenue_gbp"] - median_revenue))
scale = 1.4826 * mad if mad > 0 else daily["net_revenue_gbp"].std(ddof=0)
daily["robust_z"] = (daily["net_revenue_gbp"] - median_revenue) / max(scale, 1e-9)
daily["requires_review"] = daily["robust_z"].abs().ge(CFG.anomaly_threshold)
anomalies = daily.loc[daily["requires_review"]].sort_values("robust_z", key=np.abs, ascending=False)

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(daily["date"], daily["net_revenue_gbp"], color="#475569", linewidth=1.3)
ax.scatter(anomalies["date"], anomalies["net_revenue_gbp"], color="#dc2626", label="review flag", zorder=3)
ax.set(title="Daily net ledger value with robust review flags", xlabel="", ylabel="GBP")
ax.legend()
plt.tight_layout()
anomaly_figure = ARTIFACTS / "daily_revenue_anomalies.png"
plt.savefig(anomaly_figure, dpi=170, bbox_inches="tight")
plt.show()
display(anomalies.head(15))

## 11. Executive findings

These statements are generated from the current run so the notebook cannot silently retain stale numbers.

In [ ]:
best_month = sales_monthly.loc[sales_monthly["gross_revenue_gbp"].idxmax()]
largest_segment = segment_summary.loc[segment_summary["customers"].idxmax()]
highest_value_segment = segment_summary.loc[segment_summary["total_revenue_gbp"].idxmax()]
top_export_market = international.iloc[0]

executive_summary = {
    "data_quality": (
        f"The source contains {len(raw):,} rows. The pipeline removed "
        f"{len(cleaned['duplicates']):,} exact duplicates while retaining cancellations and returns in an auditable ledger."
    ),
    "revenue": (
        f"Completed sales generated £{gross_revenue:,.0f} gross historical revenue; "
        f"valid signed ledger value was £{net_ledger_revenue:,.0f}."
    ),
    "seasonality": (
        f"The highest gross-revenue month was {best_month['month']:%Y-%m} at "
        f"£{best_month['gross_revenue_gbp']:,.0f}."
    ),
    "customers": (
        f"The largest RFM segment was {largest_segment['segment']} ({int(largest_segment['customers']):,} customers); "
        f"the highest historical-value segment was {highest_value_segment['segment']}."
    ),
    "international": (
        f"The largest non-UK market by completed-sales revenue was {top_export_market['country']} "
        f"at £{top_export_market['revenue_gbp']:,.0f}."
    ),
    "monitoring": f"The robust daily monitor flagged {len(anomalies)} dates for investigation.",
}

for heading, finding in executive_summary.items():
    print(f"{heading.upper()}: {finding}")

## 12. Export artifacts and a reproducibility manifest

The full cleaned tables are intentionally not committed to GitHub. They are regenerated from UCI and exported during execution.

In [ ]:
raw_quality_path = ARTIFACTS / "raw_quality_profile.csv"
reconciliation_path = ARTIFACTS / "cleaning_reconciliation.csv"
monthly_path = ARTIFACTS / "monthly_performance.csv"
products_path = ARTIFACTS / "product_performance.csv"
countries_path = ARTIFACTS / "country_performance.csv"
segments_path = ARTIFACTS / "customer_segments.csv"
retention_path = ARTIFACTS / "cohort_retention.csv"
anomalies_path = ARTIFACTS / "daily_anomalies.csv"
sales_parquet_path = ARTIFACTS / "completed_sales.parquet"

raw_quality.to_csv(raw_quality_path)
reconciliation.to_csv(reconciliation_path, index=False)
sales_monthly.to_csv(monthly_path, index=False)
product_performance.to_csv(products_path, index=False)
country_performance.to_csv(countries_path, index=False)
rfm.to_csv(segments_path, index=False)
cohort_retention.to_csv(retention_path)
anomalies.to_csv(anomalies_path, index=False)
completed_sales.to_parquet(sales_parquet_path, index=False, compression="snappy")

data_card = {
    "project": "Retail Data Cleaning and Customer Analysis",
    "source": SOURCE_FINGERPRINT,
    "raw_shape": list(raw.shape),
    "headline_kpis": headline_kpis,
    "cleaning_principles": [
        "Remove only exact duplicate rows from the ledger.",
        "Keep valid cancellations and returns with signed values.",
        "Exclude anonymous customers only from customer-level analysis.",
        "Do not winsorise or delete large purchases without investigation.",
    ],
    "limitations": [
        "Historical data ending in December 2011.",
        "No product cost or margin fields, so revenue is not profit.",
        "No customer acquisition channel or demographic attributes.",
        "RFM segments are descriptive and not causal or predictive.",
        "The final monthly cohort is right-censored by the dataset end date.",
    ],
}
data_card_path = ARTIFACTS / "data_card.json"
data_card_path.write_text(json.dumps(data_card, indent=2, default=str), encoding="utf-8")

artifact_paths = [
    raw_quality_path, reconciliation_path, monthly_path, products_path,
    countries_path, segments_path, retention_path, anomalies_path,
    sales_parquet_path, data_card_path, monthly_figure, product_country_figure,
    segment_figure, cohort_figure, anomaly_figure,
]
manifest = pd.DataFrame([
    {"file": path.name, "bytes": path.stat().st_size, "sha256": sha256_file(path)}
    for path in artifact_paths
])
manifest_path = ARTIFACTS / "manifest.csv"
manifest.to_csv(manifest_path, index=False)
display(manifest)

## 13. Automated acceptance tests

These checks prevent the notebook from being presented as complete when core contracts fail.

In [ ]:
def run_acceptance_tests() -> None:
    assert list(raw.columns) == EXPECTED_COLUMNS
    assert len(raw) == 541_909
    assert ledger.duplicated().sum() == 0
    assert len(ledger) + len(cleaned["duplicates"]) == len(raw)
    assert completed_sales["quantity"].gt(0).all()
    assert completed_sales["unit_price"].gt(0).all()
    assert completed_sales["description"].notna().all()
    assert ~completed_sales["is_cancellation"].any()
    assert np.allclose(
        completed_sales["line_value_gbp"],
        completed_sales["quantity"] * completed_sales["unit_price"],
    )
    assert customer_sales["customer_id"].notna().all()
    assert rfm[["r_score", "f_score", "m_score"]].apply(lambda column: column.between(1, 5).all()).all()
    assert np.allclose(cohort_retention[1], 1.0)
    assert sales_monthly["month"].is_monotonic_increasing
    assert gross_revenue >= net_ledger_revenue
    assert manifest["sha256"].str.fullmatch(r"[0-9a-f]{64}").all()
    assert all(path.exists() and path.stat().st_size > 0 for path in artifact_paths)
    print("ALL PROJECT 01 ACCEPTANCE TESTS PASSED")


run_acceptance_tests()

## 14. Optional customer lookup application

This interface retrieves historical RFM information for a known customer identifier. It does not make automated marketing decisions.

In [ ]:
def customer_lookup(customer_id: str) -> tuple[str, pd.DataFrame]:
    value = str(customer_id).strip()
    row = rfm.loc[rfm["customer_id"].eq(value)]
    if row.empty:
        return "Customer not found in the known-customer sales table.", pd.DataFrame()
    record = row.iloc[0]
    summary = (
        f"Segment: **{record['segment']}**  \n"
        f"Recency: **{int(record['recency_days'])} days**  \n"
        f"Completed orders: **{int(record['frequency'])}**  \n"
        f"Historical revenue: **£{record['monetary_gbp']:,.2f}**"
    )
    orders_table = (
        customer_sales.loc[customer_sales["customer_id"].eq(value)]
        .groupby("invoice_no")
        .agg(invoice_date=("invoice_date", "min"), revenue_gbp=("line_value_gbp", "sum"), units=("quantity", "sum"))
        .reset_index()
        .sort_values("invoice_date", ascending=False)
        .head(20)
    )
    return summary, orders_table


try:
    import gradio as gr
    with gr.Blocks(title="Retail Customer Analysis") as app:
        gr.Markdown("# Retail Customer Analysis\nEnter a customer identifier from the dataset.")
        customer_input = gr.Textbox(label="Customer ID")
        lookup_button = gr.Button("Find customer", variant="primary")
        summary_output = gr.Markdown()
        orders_output = gr.Dataframe(label="Most recent completed orders", interactive=False)
        lookup_button.click(customer_lookup, customer_input, [summary_output, orders_output])
    if CFG.launch_app:
        app.launch(share=True, debug=False)
    else:
        print("Application built. Set launch_app=True in ProjectConfig to launch it.")
except ImportError:
    print("Gradio is unavailable. Run the dependency cell, then rerun this cell.")

known_customer = rfm.iloc[0]["customer_id"]
smoke_summary, smoke_orders = customer_lookup(known_customer)
assert "Segment:" in smoke_summary and not smoke_orders.empty
print("Customer lookup smoke test passed.")

## Interview explanation

**Problem:** The source ledger mixes completed sales with cancellations, returns, duplicates, free or adjustment rows and anonymous customers. Direct aggregation would produce inconsistent customer and revenue figures.

**Approach:** I created separate deduplicated ledger, completed-sales and known-customer views; reconciled every row; tested the rules on a controlled fixture; and built monthly KPIs, product/country analysis, RFM segmentation, cohort retention and robust anomaly monitoring.

**Engineering evidence:** The notebook records the source fingerprint, exports reproducible artifacts, hashes the outputs and runs acceptance tests over the complete 541,909-row dataset.

**CV bullet after verified execution:**

> Cleaned and reconciled 541,909 UK retail transactions using Pandas and NumPy, separating cancellations, returns, duplicates and anonymous customers; produced tested revenue KPIs, RFM segments, cohort retention, anomaly monitoring and reproducible data artifacts.

## Honest limitations

- Revenue is not profit because costs and margins are unavailable.
- Anonymous sales cannot be assigned to customer cohorts.
- RFM labels are descriptive and require commercial validation before campaign use.
- Historical relationships may not represent modern ecommerce behaviour.